# ML3 - Sentiment analysis

Implementiamo:
- un'architettura di rete neurale con almeno un layer **RNN** per il task di **sentiment analysis**
  sul dataset [`Airlines-Tweets-Sentiments`](https://www.openml.org/search?type=data&status=active&id=43397) (OpenML, id 43397);
- la fase di **training**;
- la fase di **test**.

Gli iperparametri sono impostati a valori di default (nessuna validation phase, come richiesto).

Il dataset e' la versione OpenML del noto *Twitter US Airline Sentiment* dataset: ~14.640 tweet
verso 6 compagnie aeree USA, ciascuno etichettato come `negative`, `neutral` o `positive`.

# Import

In [1]:
# torch
!pip3 install torch
# scikit-learn
!pip3 install scikit-learn
# pandas
!pip3 install pandas
# numpy
!pip3 install numpy


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\Danilo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\Danilo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\Danilo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\Danilo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import re
import random
import os
from collections import Counter
from datetime import datetime

import numpy as np
import pandas as pd

In [3]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Configurazione

In [5]:
data_dir = 'data'
save_dir = 'models'

In [6]:
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# Caricamento dei dati

Usiamo il dataset [`Airlines-Tweets-Sentiments`](https://www.openml.org/search?type=data&status=active&id=43397),
scaricato tramite l'utility [`fetch_openml`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_openml.html)
di `scikit-learn` richiedendo un DataFrame `pandas` (`as_frame=True`) perche' ci serve il **testo** grezzo del tweet, non solo feature numeriche, e dobbiamo scegliere quali feature usare: utilizzeremo solo la feature del testo e quella del sentiment come output.

Ogni esempio e' un tweet diretto a una compagnia aerea statunitense. Il target e' `tweet_sentiment_value`
(`negative` / `neutral` / `positive`); il testo del tweet e' nella colonna `tweet_text`.

Useremo questo dataset per un task di **classificazione multiclasse** (a singola etichetta): dato il testo
di un tweet, riconoscere il sentiment espresso verso la compagnia aerea.

In [7]:
#Carica il dataset dalla repository OpenML
data = fetch_openml(data_id=43397, as_frame=True, data_home=data_dir)

In [8]:
# Ricava il Datafarme
df = data.frame
print(df.shape)
print(df.columns.tolist())

(1097, 4)
['_id', 'tweet_text', 'tweet_lang', 'tweet_sentiment_value']


In [9]:
# Accediamo direttamente alle colonne che ci servono: il testo del tweet e l'etichetta di sentiment
texts = df['tweet_text'].astype(str).tolist()
labels = df['tweet_sentiment_value'].tolist()
num_labels = len(set(labels))

print('Numero di esempi:', len(texts))
print('Numero di etichette:', num_labels)
print('Distribuzione delle etichette:', Counter(labels))

Numero di esempi: 1097
Numero di etichette: 3
Distribuzione delle etichette: Counter({1: 502, 0: 406, 2: 189})


# Preprocessing del testo

Prima di dare in pasto i tweet alla rete neurale dobbiamo trasformare il testo grezzo in sequenze di id
interi (token). Procediamo in tre passi:
1. **cleaning**: eliminazione di punetggiatura e sovrascrittura dei testi in lower case (per evitare di sovraffollare il vocabolario con parole uguali ma diverse solo per punteggiatura e maiucole);
2. **tokenizzazione**: semplice separazione sugli spazi bianchi dei testi;
3. **vocabolario**: costruzione di un vocabolario (parola &rarr; id intero) usando solo il training set (per
   evitare fughe di informazione dal test set), poi mappatura di ogni tweet in una sequenza di id a
   lunghezza fissa (con padding per i tweet piu' corti, troncamento per quelli piu' lunghi).

In [10]:
# Dato un testo, lo restituisce pulito (come descritto sopra)
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z' ]+", " ", text)         # mantiene solo lettere e apostrofi
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Tokenizzazione basata sugli spazi bianchi
def tokenize(text):
    return text.split()

In [11]:
# Applicazione della pulizia
cleaned_texts = [clean_text(t) for t in texts]
# Applicazione della tokenizzazione
tokenized_texts = [tokenize(t) for t in cleaned_texts]

print(texts[0])
print(tokenized_texts[0])

this airfrance b777-300er has the oldest ifes i've ever seen. it belongs in a museum. the terrible smell isn't helping either.
['this', 'airfrance', 'b', 'er', 'has', 'the', 'oldest', 'ifes', "i've", 'ever', 'seen', 'it', 'belongs', 'in', 'a', 'museum', 'the', 'terrible', 'smell', "isn't", 'helping', 'either']


# Split Train-Test

Come indicato nella consegna, qui non serve una fase di validation, quindi facciamo un semplice split
**80-20** train-test del dataset (stratificato sull'etichetta, per mantenere le stesse proporzioni tra le
classi in entrambi gli insiemi, dato che il dataset e' sbilanciato).

In [12]:
random_state = 0
test_size = 0.2 # percentuale di dati da utilizzare per il test

In [13]:
# Train-Test split con stratificazione
x_train, x_test, y_train, y_test = train_test_split(
    tokenized_texts, labels, random_state=random_state, test_size=test_size, stratify=labels
)

print("Dimensione training set: {}".format(len(y_train)))
print("Dimensione test set: {}".format(len(y_test)))

Dimensione training set: 877
Dimensione test set: 220


# Vocabolario e codifica in tensori

Costruiamo il vocabolario usando solo i token del **training**, poi codifichiamo ogni tweet (train e test) come una sequenza a lunghezza fissa (`max_len`) di
id interi, usando un id speciale `<pad>` per il padding e un id `<unk>` per le parole fuori vocabolario.

In [14]:
max_len = 25  # Numero massimo di token mantenuti per testo, i tweet non sono troppo lunghi

# parole_uniche = set di parole uniche presenti nel training set (x_train):
# per ogni token nel training set:
#   per ogni parola nel token:
#       aggiungi la parola all'insieme parole_uniche
parole_uniche = sorted(set(parola for tokens in x_train for parola in tokens))

# Costruzione del vocabolario: assegniamo un id intero univoco a ogni parola diversa vista nel training set
vocab = {'<pad>': 0, '<unk>': 1} # intero 0 = padding; intero 1 = parola sconosciuta (non vista nel training set)
for parola in parole_uniche:
    vocab[parola] = len(vocab) # assegna un id univoco a ogni parola, in ordine di come le trova (gli assegna la lunghezza del vocabolario)

vocab_size = len(vocab)
print('Dimensione del vocabolario:', vocab_size)

Dimensione del vocabolario: 3481


In [15]:
# Mappa una lista di token in una lista a lunghezza fissa di id interi
def encode(tweet, vocab, max_len):
    ids = [vocab.get(token, vocab['<unk>']) for token in tweet][:max_len] # per ogni token: prendi l'intero corrispondente nel vocabolario, e se non esiste usa <unk>; poi taglia la lista a max_len
    ids = ids + [vocab['<pad>']] * (max_len - len(ids)) # Padding alla fine
    return ids # Lista di intero che codifica i token, con lunghezza fissa max_len

# Variabili finali
# Un tweet (= elemento di x_train) diventa una lista di interi
x_train_ids = [encode(tweet, vocab, max_len) for tweet in x_train]
x_test_ids = [encode(tweet, vocab, max_len) for tweet in x_test]

print(x_train[0])
print(x_train_ids[0])

['airfrance', 'can', 'i', 'not', 'have', 'a', 'duo', 'seat', 'like', 'i', 'said', "it's", 'only', 'fair', 'since', 'my', 'hassle', 'on', 'my', 'inward', 'journey']
[99, 480, 1432, 2089, 1317, 8, 874, 2652, 1755, 1432, 2611, 1551, 2163, 1037, 2731, 1989, 1310, 2158, 1989, 1535, 1608, 0, 0, 0, 0]


# Tensori e Dataset

Definiamo delle funzioni per trasformare i nostri dati in **Tensori** (procedura standard per **PyTorch**):

In [16]:
# Dati features e labels, restituisce il dataset convertito in Tensore
def get_tensor_dataset(x_ids, y):
    x_tensor = torch.LongTensor(x_ids)
    y_tensor = torch.LongTensor(y)

    tensor_dataset = torch.utils.data.TensorDataset(x_tensor, y_tensor)
    return tensor_dataset

In [17]:
# Trasformiamo training e test set in Tensore
train_dataset = get_tensor_dataset(x_train_ids, y_train)
test_dataset = get_tensor_dataset(x_test_ids, y_test)

In [18]:
# Vado avanti 32 tweet per volta con la discesa del gradiente
batch_size = 32

# Dividiamo training e test set in batch (messcolando il training set)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

# Rete neurale

### 1. Iperparametri

Definiamo alcuni **iperparametri**:
- **batch_size**: gia' impostato sopra, numero di esempi per batch.
- **embed_dim**: dimensione degli **embedding*** delle parole appresi dalla rete;
- **hidden_dim**: numero di unita' dello stato nascosto della **RNN**.
- **lr**: **learning rate** (tasso di apprendimento).
- **num_epochs**: numero di **epoche** di training.

Come richiesto, sono impostati a valori di default ragionevoli (nessuna fase di validation/tuning).

>**embedding*** : all'inizio avevamo dei testi come valori in input; li abbiamo puliti, tokenizzati, abbiamo creato un vocabolario per convertire parole in int e abbiamo trasformato ogni tweet in input in una serie di interi. Ma una rete neurale non è n grado di comprendere il significato di questi interi, dunque serve uno strato di **Embedding** con cui ogni parola nel vocabolario avrà un vettore di valori lungo **embed_dim** di pesi che saranno addestrati: parole semanticamente vicine avranno vettori con differenza minima; mentre parole semanticamente lontane avranno vettori di differenza più alta. 

In [19]:
# batch_size = 32
embed_dim = 64 # Ogni parola avrà un vettore di 64 interi da addestrare sulla semantica
hidden_dim = 64
lr = 0.001
num_epochs = 17

seed = 10  # seed fisso per tutte le scelte casuali in PyTorch
log_every = 1  # logging ogni x epoche

In [20]:
# Setting del seed per rendere riproducibili le azioni casuali
def set_seed(seed):
    torch.manual_seed(seed)
    random.seed(seed)

### 2. Definizione dell'architettura della rete neurale

La rete e' composta da tre livelli (layer):
1. un **layer di embedding** (`nn.Embedding`), che trasforma ogni id di parola in un vettore denso apprendibile, è come una tabella che la rete impara ad aggiornare, dove parole con significato simile finiscono per avere vettori simili; 
2. un **layer RNN** (`nn.RNN`, `batch_first=True`), che elabora la sequenza di embedding e produce uno
   stato nascosto finale che riassume l'intero tweet; l'RNN scorre la sequenza un token alla volta, mantenendo una "memoria" (hidden state) che aggiorna ad ogni passo; 
3. un **layer completamente connesso (lineare)** (`nn.Linear`) sopra lo stato nascosto finale della RNN,
   che produce uno score (logit) per ciascuna classe di sentiment.

In [21]:
class RNNSentiment(nn.Module):

    # inizializzazione della RNN
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_labels, pad_idx=0):
        super(RNNSentiment, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx) # Strato 1
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True) # Starto 2
        self.fc = nn.Linear(hidden_dim, num_labels) # Strato 3

    # Funzione di passaggio tra un livello e l'altro della RNN con input x
    def forward(self, x): 
        emb = self.embedding(x) # Applicazione strato 1
        _, hidden = self.rnn(emb) # Applicazione strato 2
        hidden = hidden.squeeze(0)
        out = self.fc(hidden)  # Applicazione strato 3
        
        return out

### 3. Modello, device, funzione di perdita e optimizer

In [22]:
# Istanziazione del modello
model = RNNSentiment(vocab_size, embed_dim, hidden_dim, num_labels)

# scelta del device (GPU se disponibile, altrimenti CPU)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device: {}".format(device))

model.to(device)  # Passa il modello al device corrente

Device: cpu


RNNSentiment(
  (embedding): Embedding(3481, 64, padding_idx=0)
  (rnn): RNN(64, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=3, bias=True)
)

In [23]:
# Stampa dei parametri addestrabili del modello: ca. 200000 sono molti per un training set di ca. 800 istanza (rischio Overfitting)
tot = 0
for name, weight in model.named_parameters():
    print(name, weight.shape)
    tot += len(weight.flatten())
print('Numero totale di parametri: ' + str(tot))

embedding.weight torch.Size([3481, 64])
rnn.weight_ih_l0 torch.Size([64, 64])
rnn.weight_hh_l0 torch.Size([64, 64])
rnn.bias_ih_l0 torch.Size([64])
rnn.bias_hh_l0 torch.Size([64])
fc.weight torch.Size([3, 64])
fc.bias torch.Size([3])
Numero totale di parametri: 231299


In [24]:
# Funzione di perdita: Cross-EntropyLoss è adatta per problemi di classificazione multi-classe, come il nostro caso con 3 etichette di sentiment.
criterion = nn.CrossEntropyLoss()

# Ottimizzatore: Adam, converge subito con poche epoche
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# Setting del seed precedentemente definito
set_seed(seed)

# Training

Addestramento del modello (procedura standard). 

In [25]:
stats = {metric: [] for metric in ['loss', 'acc']}
# per ogni epoca:
for epoch in range(1, num_epochs + 1):

    # Resetta le metriche
    train_tot, train_epoch_loss, train_epoch_acc = 0, 0.0, 0
    model.train(True)  # modalita' training: attiva il tracciamento dei gradienti
    # per ogni batch:
    for train_inputs, train_labels in train_loader:
        # passa il batch al device corrente
        train_inputs, train_labels = train_inputs.to(device), train_labels.to(device)

        # azzera i gradienti dei parametri dai batch precedenti (altrimenti si accumulano)
        optimizer.zero_grad()

        # forward propagation e calcolo della loss
        train_preds = model(train_inputs)
        train_loss = criterion(train_preds, train_labels)

        # backward propagation e passo di discesa del gradiente
        train_loss.backward()
        optimizer.step()

        # applica la funzione softmax e prende l'etichetta con probabilita' piu' alta (per calcolare le metriche)
        train_preds = torch.log_softmax(train_preds, dim=1)
        _, train_preds = torch.max(train_preds, dim=1)

        # calcola l'accuratezza
        train_acc = torch.sum(train_preds == train_labels.data)

        # aggiorna loss e accuratezza dell'epoca
        train_epoch_loss += train_loss.item() * train_inputs.size(0)
        train_epoch_acc += train_acc.item()
        train_tot += train_inputs.size(0)

    # aggiorna le metriche
    train_loss = train_epoch_loss / train_tot
    train_acc = train_epoch_acc / train_tot
    stats['loss'].append(train_loss)
    stats['acc'].append(train_acc)

    if epoch % log_every == 0:
        now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        s = "[{}] Epoca {} | Train Loss: {:.4f} | Train Acc: {:.4f}"
        print(s.format(now, epoch, train_loss, train_acc))

[2026-07-08 10:22:06] Epoca 1 | Train Loss: 1.0456 | Train Acc: 0.4299
[2026-07-08 10:22:06] Epoca 2 | Train Loss: 1.0084 | Train Acc: 0.4960
[2026-07-08 10:22:06] Epoca 3 | Train Loss: 0.9716 | Train Acc: 0.5371
[2026-07-08 10:22:07] Epoca 4 | Train Loss: 0.9001 | Train Acc: 0.5929
[2026-07-08 10:22:07] Epoca 5 | Train Loss: 0.8376 | Train Acc: 0.6454
[2026-07-08 10:22:07] Epoca 6 | Train Loss: 0.8100 | Train Acc: 0.6682
[2026-07-08 10:22:07] Epoca 7 | Train Loss: 0.7280 | Train Acc: 0.6967
[2026-07-08 10:22:07] Epoca 8 | Train Loss: 0.6569 | Train Acc: 0.7332
[2026-07-08 10:22:08] Epoca 9 | Train Loss: 0.6054 | Train Acc: 0.7719
[2026-07-08 10:22:08] Epoca 10 | Train Loss: 0.5752 | Train Acc: 0.7868
[2026-07-08 10:22:08] Epoca 11 | Train Loss: 0.5901 | Train Acc: 0.7708
[2026-07-08 10:22:08] Epoca 12 | Train Loss: 0.5563 | Train Acc: 0.7868
[2026-07-08 10:22:08] Epoca 13 | Train Loss: 0.5148 | Train Acc: 0.7993
[2026-07-08 10:22:08] Epoca 14 | Train Loss: 0.4446 | Train Acc: 0.8290
[

### Salvataggio del modello

In [26]:
model_path = os.path.join(save_dir, 'rnn_sentiment.pth')
torch.save(model.state_dict(), model_path)

# Test

In [27]:
# Prima carichiamo il modello salvato
model_reloaded = RNNSentiment(vocab_size, embed_dim, hidden_dim, num_labels)
resume = torch.load(model_path, map_location=device)
model_reloaded.load_state_dict(resume)
model_reloaded.to(device)
model_reloaded.eval() # Modalita' evaluation: disattiva il tracciamento dei gradienti

RNNSentiment(
  (embedding): Embedding(3481, 64, padding_idx=0)
  (rnn): RNN(64, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=3, bias=True)
)

In [28]:
# Testing (stessa procedura del training, ma senza calcolo dei gradienti e senza aggiornamento dei pesi)
# per ogni batch (1 solo in test): 
for batch in test_loader:
    with torch.no_grad():
        test_inputs, test_labels = batch
        test_inputs, test_labels = test_inputs.to(device), test_labels.to(device)
        test_preds = model_reloaded(test_inputs)

        test_preds = torch.log_softmax(test_preds, dim=1)
        _, test_preds = torch.max(test_preds, dim=1)
        test_acc = torch.sum(test_preds == test_labels.data)

    print('Accuratezza sul test set: ' + str(test_acc.item() / len(test_labels)))

Accuratezza sul test set: 0.5181818181818182
